# Semaine 10 — Jour 4 : NER, offsets et masquage contrôlé

Reconnaître des entités de service, technologie, environnement et email.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour l’unité complète avec entretien et challenge. L’installation de l’environnement ajoute un temps distinct; le lab LoRA est optionnel.

Prérequis : Python 3.11+, bases de classification et de vecteurs. Le parcours utilise NumPy et scikit-learn, sans modèle à télécharger ni clé API. Installer les dépendances S10 selon [le guide](../../../docs/week10/QUICKSTART.md).

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Tenter [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks depuis la racine : `notebooks/week10/S10_J4_ner.ipynb` et `notebooks/week10/teacher/S10_J4_ner_teacher.ipynb`.

Critère de réussite : Masquer les occurrences reconnues sans promettre une anonymisation complète.

Les apprentissages du parcours principal sont réellement exécutables. Le corpus reste fictif, les baselines à règles sont identifiées et aucun résultat LoRA n’est annoncé.


# Objectifs mesurables

- Reconnaître des entités de service, technologie, environnement et email.
- Évaluer les spans exacts avec leur type et leur document.
- Masquer les occurrences reconnues sans promettre une anonymisation complète.

## Preuves attendues

Une sortie reproductible, un invariant vérifié et une limite de la tâche correctement interprétée.


# S10 J4 — NER, offsets et masquage contrôlé


## 1. NER et extraction structurée
La reconnaissance d’entités nommées repère des occurrences typées dans un texte. L’extraction J2 agrège des occurrences vers un schéma métier avec valeurs normalisées et ambiguïtés. Plusieurs entités SERVICE peuvent être reconnues sans savoir laquelle est responsable de l’incident. Les tâches se complètent mais ne se substituent pas l’une à l’autre.

Le lab utilise un lexique de services et technologies ainsi qu’une regex email. Il n’entraîne pas de modèle NER neuronal. Le choix est explicite pour concentrer l’expérience obligatoire sur frontières de spans, collisions et évaluation. Une méthode apprise demanderait annotations, entraînement et comparaison sur des entités inconnues.

## 2. Contrat de span
Chaque entité contient start, end, text et label. Les indices sont des positions de caractères Python dans le texte original, avec end exclusif. Les accents et la casse originale restent conservés, même si le motif utilise une recherche insensible à la casse. Une réponse d’API ou une interface Java doit préciser l’unité : les offsets UTF-16 ne sont pas toujours les indices Unicode Python.

L’égalité text == source[start:end] est nécessaire. Le typage et la frontière exacte comptent aussi : reconnaître seulement Paiement au lieu de Paiement API ne réussit pas le protocole strict. Les quatre exemples sont annotés explicitement sans appeler le détecteur pour fabriquer leur gold.

## 3. Chevauchements et règles
Les candidats sont triés par longueur décroissante et une sélection évite les chevauchements. Les entités retenues sont remises dans l’ordre source. Le lexique protège les limites de mots pour éviter de reconnaître prod à l’intérieur de production ou Redis dans une chaîne plus longue.

Cette politique privilégie la simplicité et un span long. Elle ne gère pas des entités imbriquées. D’autres corpus pourraient avoir besoin de plusieurs couches d’annotation, d’un schéma BIO par token ou d’un modèle spaCy/Transformer. L’évolution du schéma fait partie de la tâche, pas seulement du choix de bibliothèque.

## 4. Évaluation stricte
Une correspondance exige même document, début, fin et type. Un mauvais type compte comme une prédiction incorrecte et laisse une entité gold manquante. Les métriques précision, rappel et F1 sont calculées sur les ensembles de spans. Les cas sans prédiction ne reçoivent pas automatiquement un score parfait.

Le lexique réussit les noms connus du petit corpus, ce qui ne mesure pas la généralisation. Un nouveau service inconnu peut être totalement omis. Ajouter au jeu réservé des noms absents du lexique, des variantes et des mots ordinaires ressemblant à des entités. Ne pas seulement mesurer les formes qui ont servi à écrire les règles.

## 5. Masquage
mask_entities remplace les spans depuis la fin vers le début afin que les premières positions ne soient pas décalées pendant les remplacements. Il vérifie bornes, texte et absence de chevauchement avant de modifier le résultat. Les labels remplacent les mentions reconnues; ils ne produisent pas un identifiant de pseudonyme stable.

Ce masquage est partiel. Une personne, un secret ou une adresse écrite autrement peut ne pas être reconnue. Il n’établit aucune anonymisation juridique ou technique générale. Les offsets du texte original ne doivent pas être réutilisés comme s’ils pointaient dans le texte masqué.

## 6. Lecture des négations
« Ne pas utiliser Redis » contient bien une mention de technologie Redis. La NER ne signifie pas que Redis est utilisé, responsable ou autorisé. Pour produire cette conclusion, il faudrait une analyse de relation et de polarité. La reconnaissance d’une chaîne ne doit pas déclencher une action métier.

## À retenir
Les frontières, types et conventions d’offsets font partie du contrat. Un bon F1 sur des noms connus ne prouve pas un masquage exhaustif des données sensibles.


## Diagramme du jour

```mermaid
flowchart TD
 A["Texte original"] --> B["Candidats de spans"]
 B --> C["Résolution des chevauchements"]
 C --> D["Spans typés exacts"]
 D --> E["Métriques par document"]
 D --> F["Masquage partiel"]
```

Le rendu Mermaid dépend du lecteur; la source .mmd reste disponible.

# Lab exécutable

NumPy et scikit-learn sont requis; aucune requête réseau dans les cellules.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week10/nlp.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
from examples.week10.nlp import read_data, entities, span_metrics, mask_entities
examples = read_data("examples.json")
predicted = [{"id":r["id"], "entities":entities(r["text"])} for r in examples]
metrics = span_metrics(examples, predicted)
assert metrics["true_positive"] == 12
for row in examples:
    assert all(row["text"][s["start"]:s["end"]] == s["text"] for s in entities(row["text"]))
print(metrics)
print(mask_entities(examples[0]["text"], predicted[0]["entities"]))

# Lab J4 — NER, offsets et masquage contrôlé

Depuis la racine du dépôt, dans l’environnement S10 :

```bash
python3 book/week10/day04/labs/run.py
python3 -m examples.week10.demo --day 4
```

Résultat attendu : **12 spans connus évalués exactement sur quatre textes**.

## Travail

1. Exécuter le scénario et identifier la nature de chaque composant : appris, adapté ou à règles.
2. Réaliser l’exercice 2 sur une copie, sans modifier le corpus distribué.
3. Expliquer le dénominateur du résultat et une limite.

Les données sont sous `examples/week10/data`; le code commun sous `examples/week10/nlp.py`; les tests sous `tests/week10`. Lire [QUICKSTART](../../../../docs/week10/QUICKSTART.md) pour les dépendances, artefacts et commandes.

Les modèles entraînés sont conservés en mémoire dans chaque processus pour accélérer les notebooks. Une modification des données demande un nouveau processus ou de vider explicitement les caches trained/specialized avant de réentraîner. Les correctifs du notebook enseignant s’exécutent après le lab élève.


# Exercices

Conserver une première réponse avant les corrigés.

## Exercice 1 — Comparer deux spans

Le gold porte SERVICE sur Paiement API, la prédiction SERVICE sur Paiement seulement. Comment compter ?

Livrable : réponse argumentée avec calcul, sortie ou test.

## Exercice 2 — Relier le négatif

Redis dans Ne pas utiliser Redis est-il une entité ? Que ne prouve-t-elle pas ?

Livrable : réponse argumentée avec calcul, sortie ou test.

## Exercice 3 — Masquer sans décalage

Pourquoi appliquer les remplacements de droite à gauche ?

Livrable : réponse argumentée avec calcul, sortie ou test.


# Entretien

Répondre en deux minutes par question avec un exemple et une limite.

## Question 1

Quel avantage et quelle limite d’un lexique ?

## Question 2

Comment comparer une NER apprise à cette baseline ?


# Challenge

Masquer les entités connues d’un exemple, puis montrer qu’un nouveau nom de service absent du lexique n’est pas nécessairement détecté. Expliquer pourquoi ce résultat ne permet pas d’annoncer une anonymisation complète.

## Critères

Protocole reproductible, invariant explicite et interprétation honnête du résultat, y compris sans amélioration.


# Références — Jour 4

Sources primaires consultées le 20 octobre 2026. Le corpus et les annotations du lab sont fictifs; les seuils sont pédagogiques.

- [scikit-learn 1.8 : pièges et fuite de données](https://scikit-learn.org/1.8/common_pitfalls.html).
- [SGDClassifier 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.linear_model.SGDClassifier.html) — log_loss et partial_fit.
- [TfidfVectorizer 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).
- [Neighborhood Components Analysis 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.neighbors.NeighborhoodComponentsAnalysis.html) — apprentissage métrique supervisé.
- [spaCy : entités nommées](https://spacy.io/usage/linguistic-features#named-entities) — vocabulaire et conventions; spaCy n’est pas utilisé dans le lab obligatoire.
- [PEFT : LoRA](https://huggingface.co/docs/peft/package_reference/lora) — extension Transformer optionnelle, non exécutée.

Lire en priorité les références du thème étudié. Les concepts communs à plusieurs bibliothèques ne rendent pas leurs objets ou artefacts interchangeables. Les dépendances obligatoires correspondent à l’environnement testé, les dépendances LoRA restent à verrouiller pour l’expérience optionnelle.
